In [0]:
%sql
CREATE OR REPLACE VIEW bronze_EDA AS
SELECT
			CASE 
				WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
				WHEN edl.precio::float BETWEEN 0 AND 2147483648 THEN edl.precio::float 
				ELSE NULL
			END AS precio,

			CASE 
				WHEN edl.numero = 'NaN' or edl.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
				WHEN edl.numero::float BETWEEN 0 and 50000 THEN edl.numero::float
				ELSE NULL
			END AS numero,
			edl.calle,

			CASE
				WHEN edl.expensas = 'NaN' or edl.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
				WHEN edl.expensas::float BETWEEN 0 AND 20000000 THEN edl.expensas::float
				ELSE NULL
			END AS expensas,
			edl.tipo_de_operacion,

			CASE
				WHEN lower(edl.moneda) like '%dolares%' THEN 'USD'
				WHEN lower(edl.moneda) like '%us%'      THEN'USD'
				WHEN lower(edl.moneda) like '%pesos%'   THEN'ARS'
				WHEN lower(edl.moneda) like '%ars%'     THEN 'ARS'
				ELSE edl.moneda
			END AS moneda,

			CASE
				WHEN edl.ambientes = 'NaN' or edl.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
				else edl.ambientes::float
			END AS ambientes,

			CASE
				WHEN edl.metros_cuadrados_totales = 'Nan' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
				ELSE edl.metros_cuadrados_totales::decimal
			END metros_cuadrados_totales,

			CASE 
				WHEN edl.metros_cuadrados_cubiertos = 'Nan' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
				ELSE edl.metros_cuadrados_cubiertos::decimal
		  END AS metros_cuadrados_cubiertos,

			--edl.orientacion_cardinal,
			edl.orientacion_inmueble,

		  CASE
				WHEN edl.piso IS NULL THEN NULL
				WHEN edl.piso = 'NaN' THEN NULL
				ELSE edl.piso::float
			END AS piso,

			CASE
				WHEN edl.cochera = 'tiene' THEN 1
				ELSE NULL
			END AS cochera,

			antiguedad,
			edl.estado,
			edl.tipo_vendedor,
			edl.original_text,
			edl.url,
			edl.zona,
			COALESCE(TRY_CAST(edl.fecha AS DATE), CURRENT_DATE()) AS fecha
FROM bootcamp_matias.raw.propiedades_bronze edl
WHERE url RLIKE 'https'

In [0]:
%sql
--Ranking de zonas por precio
WITH precio_zona AS (
  SELECT
    zona,
    moneda,
    ROUND(AVG(precio),2) AS precio_promedio,
    ROUND(PERCENTILE(precio, 0.5),2) AS mediana,
    ROUND(MAX(precio),2) AS precio_maximo,
    ROUND(MIN(precio),2) AS precio_minimo,
    COUNT(*) AS cantidad
  FROM bronze_EDA
  WHERE 
    precio > 0
    AND zona IS NOT NULL
    AND moneda IS NOT NULL
    AND tipo_de_operacion = 'alquiler'
  GROUP BY 
    zona, moneda
),
ranking_zonas AS (
  SELECT
  zona,
  moneda,
  precio_promedio,
  mediana,
  precio_maximo,
  precio_minimo,
  cantidad,
  ROW_NUMBER() OVER (PARTITION BY moneda ORDER BY precio_promedio DESC) AS rn
FROM precio_zona
)
SELECT
  rn,
  zona,
  moneda,
  precio_promedio,
  mediana,
  precio_maximo,
  precio_minimo,
  cantidad
FROM ranking_zonas
ORDER BY moneda, rn


In [0]:
%sql
--Analisis de precio promedio por m2
WITH propiedades_m2 AS(
SELECT
  zona,
  moneda,
  precio,
  ROUND(precio/metros_cuadrados_totales,2) AS precio_m2
FROM bronze_EDA
WHERE 
  precio > 0
  AND moneda ='ARS'
  AND tipo_de_operacion = 'alquiler'
  AND metros_cuadrados_totales > 10
),
precio_m2_promedio AS (
  SELECT
    zona,
    COUNT(*) AS cantidad,
    ROUND(AVG(precio_m2),2) AS precio_m2_promedio,
    ROUND(PERCENTILE(precio_m2, 0.5),2) AS mediana
  FROM propiedades_m2
  GROUP BY
    zona
   
),
ranking_zonas AS (
SELECT
  zona,
  cantidad,
  precio_m2_promedio,
  mediana,
  RANK() OVER(ORDER BY precio_m2_promedio DESC) AS rank_caro,
  RANK() OVER(ORDER BY precio_m2_promedio DESC) AS rank_barato
FROM precio_m2_promedio
)
SELECT 
  rank_caro,
  zona,
  cantidad,
  precio_m2_promedio,
  mediana
FROM ranking_zonas
ORDER BY rank_caro
LIMIT 15
  

In [0]:
%sql
WITH propiedades_vs_zonas (
  SELECT
    zona,
    moneda,
    ambientes,
    precio,
    ROUND(AVG(precio) OVER(PARTITION BY zona,moneda),2) AS precio_promedio_zona,
    COUNT(*) OVER(PARTITION BY zona, moneda) AS propiedades
  FROM bronze_EDA
  WHERE 
    precio > 0
    AND zona IS NOT NULL
    AND moneda = 'ARS'
    AND tipo_de_operacion='alquiler'
),
con_diferencia AS (

  SELECT
    zona,
    ambientes,
    precio,
    precio_promedio_zona,
    propiedades,
    ROUND(precio - precio_promedio_zona,2) AS diferencia_vs_promedio,
    ROUND((precio -precio_promedio_zona) * 100.0  / precio_promedio_zona, 2) AS pct_diferencia
  FROM propiedades_vs_zonas
  WHERE propiedades >= 20
)
SELECT
  zona,
  ambientes,
  precio,
  precio_promedio_zona,
  propiedades,
  diferencia_vs_promedio,
  pct_diferencia || '%' AS pct_diferencia
FROM con_diferencia
ORDER BY pct_diferencia DESC
LIMIT 20